# **<font color="red">MCP: Model Context Protocol</font>**
- The MCP is an open standard designed to standardize how LLM like Gemini and Claude communicate with external applications, data sources, and tools. Think of it as a universal connection mechanism that simplifies how LLMs obtain context, execute actions, and interact with various systems.

## **How does MCP work?** 
- MCP follows a client-server architecture, defining how data (resources), interactive templates (prompts), and actionable functions (tools) are exposed by an MCP server and consumed by an MCP client (which could be an LLM host application or an AI agent).

## **MCP Tools in ADK**
- ADK helps you both use and consume MCP tools in your agents, whether you are trying to build a tool to call an MCP service, or exposing an MCP server for other developers or agents to interact with your tools.
- See _Tools_ and _Integrations_ for pre-built MCP tools you can use in your agents. Refer to the _MCP Tools Documentation_ for code samples and design patterns that helps you use ADK together with MCP servers, including:
  - **Using Existing MCP Servers within ADK:** An ADK agent can act as an MCP client and use tools provided by external MCP Servers.
  - **Exposing ADK Tools via an MCP Server:** How to build an MCP server that wraps ADK tools, making them accessible to any MCP client.

## **ADK Agent and FastMCP Server**
- ADK uses *FastMCP* to handle all the complex MCP protocol details and server management, so you can focus on building great tools. It's designed to be high-level and Pythonic, in most cases, decorating a function is all you need.
- Refer to the _MCP Tool Documentation_ to how you can use ADK together with the FastMCP server running on Cloud Run.

## **MCP Servers for Google Cloud Genmedia**
- **MCP Tools for Genmedia Services** is a set of open-source MCP servers that enable you to integrate Goolge Cloud generative media services - such as ***Imagen, Veo, Chirp 3 HD Voices, and Lyria*** into your AI applications.
- **Agent Development Kit (ADK)** and **Genkit** provide built-in support for these MCP tools, allowing your AI agents to effectively orchestrate generative media workflows.

# **<font color="blue">Model Context Protocol Tools</font>**
- This guide walks you through two ways of integrating MCP with ADK.

### **Prerequisites**
1. Set up ADK
2. Install/Update Python/Java
3. Setup Node.js and npx: (Python only)
4. Verify Installations: `adk` and `npx`
  

## **<font color="green">Using MCP Servers with ADK Agents (ADK as an MCP Client) in `adk web`</font>**
- This section demonstrates how to integrate tools from external MCP servers into your ADK agents.
- This is the most common integration pattern when your ADK agent needs to use capabilities provided by an existing service that exposes an MCP interface.
- You will see how the `McpToolset` class can be directly added to your agent's `tools` list, enabling seamless connection to an MCP server, discovery of its tools, and making them available for your agent to use.
- These examples primarily focus on interactions within the `adk web` development environment.

### **`McpToolset` class**
- The `McpToolset` class is ADK's primary mechanism for integrating tools from an MCP server.
- When you include an `McpToolset` instance in your agent's `tools` list, it automatically handles the interaction with the specified MCP server. Here's how it works:
  1. **Connection Management:**
     - On initialization, `McpToolset` establishes and manages the connection to the MCP server.
     - This can be a local server process (using `StdioConnectionParams` for communication over standard input/output) or a remote server (using `SseConnectionParams` for Server-Sent Events).
     - The toolset also handles the graceful shutdown of this connection when the agent or application terminates.
  2. **Exposure to Agent:**
     - These adapted tools are then made available to your `LlmAgent` as if they were native ADK tools. 
  3. **Proxying Tool Calls:**
     - When your `LlmAgent` decides to use one of these tools, `McpToolset` transparently proxies the call (using the `call_tool` MCP method) to the MCP server, sends the necessary arguments, and returns the server's response back to the agent.
  4. **Filtering (Optional):** You can use the `tool_filter` parameter when creating an `McpToolset` to select a specific subset of tools from the MCP server, rather than exposing all of them to your agent.
- The following examples demonstrate how to use `McpToolset` within the `adk web` development environment. For scenarios where you need more fine-grained control over the MCP connection lifecycle or are not using `adk web`, refer to the "Using MCP Tools in your own Agent out of `adk web` section later in this page.


#### **Example 1: File System MCP Server**
- This Python example demonstrates connecting to a local MCP server that provides file system operations.
- **Step 1: Define your Agent with `McpToolset`**
  - Create an `agent.py` file. The `McpToolset` is instantiated directly withing the `tools` list of your `LlmAgent`.
    - **Important:** In the `args` list with the **absolute path** to an actual folder on your local system that the MCP server can access.
    - **Important:** Place the `.env` file in the parent directory of the `./adk_agent_samples` directory
  - **Step 2: Create an `__init__.py` file**
    - Ensure you have an `__init__.py` in the same directory as `agent.py` to make it a discoverable Python package for ADK.
    - ```python
    from . import agent
    ```
  - **Run `adk web` and Interact**
    - Navigate to the parent directory of `mcp_agent` in your terminal and run:
      ```cmd
      cd ./adk_agent_sample
      adk web
      ```

In [2]:
import os
from google.adk.agents import LlmAgent
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from mcp import StdioServerParameters
from config import config

# -----------------------------
# Setup API Key and model
# -----------------------------
APP_NAME = "chatbot_mcp"
USER_ID = "user_001"
SESSION_ID = "session_001"
MODEL = config.MODEL

os.environ["GOOGLE_API_KEY"] = config.GOOGLE_API_KEY.strip()

# -----------------------------
# Target folder for filesystem MCP
# -----------------------------
TARGET_FOLDER_PATH = r"D:\Agent-Development-Kit\adk22\Vikas"

# -----------------------------
# Root Agent
# -----------------------------
root_agent = LlmAgent(
    model=MODEL,
    name="filesystem_assistant_agent",
    instruction="Help the user manage their files. You can list, read, write, delete and search files in the allowed directory.",
    tools=[
        McpToolset(
            connection_params=StdioConnectionParams(
                server_params=StdioServerParameters(
                    command="npx",
                    args=[
                        "-y",
                        "@modelcontextprotocol/server-filesystem",
                        os.path.abspath(TARGET_FOLDER_PATH),
                    ],
                ),
            ),
            tool_filter=[
                "list_directory",
                "read_file",
                "write_file",
                "delete_file",
                "search_files",
            ],
        )
    ],
)


#### **Example 2: Google Maps MCP Server**
- This example demostrates connecting to the Google Maps MCP server.
- **Step 1: Get API key and Enable APIs**
  1. **Google Maps API Key:** Follow the directions at Use API Keys to obtain a Google Maps API Key.
  2. **Enable APIs:** In your Google Cloud project, ensure the following APIs are enabled:
     - Direction API
     - Routes API For instructions, see the Getting started with Google Maps Platform documentation.
- **Step 2: Define your Agent with `McpToolset` for Google Maps**
  - Create `agent.py` file.

In [ ]:
# agent.py
import os
from google.adk.agents import LlmAgent
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from mcp import StdioServerParameters
from .config import config

# -----------------------------
# Setup API Key and model
# -----------------------------
APP_NAME = "chatbot_google_map"
USER_ID = "user_001"
SESSION_ID = "session_001"
MODEL = config.MODEL

google_maps_api_key = config.GOOGLE_MAPS_API_KEY.strip()
google_api_key = config.GOOGLE_API_KEY.strip()

# set environment variables
os.environ["GOOGLE_API_KEY"] = google_api_key
os.environ["GOOGLE_MAPS_API_KEY"] = google_maps_api_key

# -----------------------------
# Create the root agent
# -----------------------------
root_agent = LlmAgent(
    model=MODEL,
    name="maps_assistant_agent",
    instruction="Help the user with mapping directions, and finding places using Google Map tools.",
    tools=[
        McpToolset(
            connection_params=StdioConnectionParams(
                server_params=StdioServerParameters(
                    command="npx",
                    args=["-y", "@modelcontextprotocol/server-google-maps"],
                    env={"GOOGLE_MAPS_API_KEY": google_maps_api_key}
                ),
            ),
            # tool_filter=[
            #     'get_directions',
            #     'find_place_by_id',
            #     'find_places',
            #     'geocode_address',
            #     'reverse_geocode',
            #     'get_place_details',
            #     'get_elevation',
            #     'calculate_distance',
            # ]
        )
    ],
)



## **<font color="green">Building an MCP Server with ADK tools (MCP Server Exposing ADK)</font>**
- This pattern allows you to wrap existing ADK tools and make them available to any standard MCP client application. The example in this section exposes the ADK `load_web_page` tool through a custom-built MCP server.
- **Summary of Steps:** You will create a standard Python MCP server application using the `mcp` library. Within this server, you will:
  1. Instantiate the ADK tool(s) you want to expose (e.g., `FunctionTool(load_web_page)`).
  2. Implement the MCP server's `@app.list_tools()` handler to advertise the ADK tool(s). This involves converting the ADK tool definition to the MCP schema using the `adk_to_mcp_tool_type` utility from `google.adk.tools.mcp_tool.conversion_utils`.
  3. Implement the MCP Server's `@app.call_tool()` handler. This handler will:
     - Receive tool call requests from MCP clients.
     - Identify if the request targets one of your wrapped ADK tools.
     - Execute the ADK tool's `.run_async()` method.
     - Format the ADK tool's result into an MCP-compliant response (e.g., `mcp.types.TextContent`.)
- **Prerequisites:** Install the MCP server library in the same Python environment as your ADK installation.
  `pip install mcp`

### **Steps**
- **Step 1: Create the MCP Server Script:** Create a new Python file for your MCP Server, for example, `my_adk_mcp_server.py`
- **Step 2: Implement the Server Logic:** Add the follwing code to `my_adk_mcp_server.py`. This script sets up and MCP server that exposes the ADK `load_web_page` tool.

In [3]:
# my_adk_mcp_server.py
import asyncio
import json

# ----------------------------------------------------
# MCP Server Imports
# ----------------------------------------------------
from mcp import types as mcp_types  # Alias to avoid naming conflicts
from mcp.server.lowlevel import Server, NotificationOptions
from mcp.server.models import InitializationOptions
import mcp.server.stdio  # Run server over STDIO transport

# ----------------------------------------------------
# ADK Tool Imports
# ----------------------------------------------------
from google.adk.tools.function_tool import FunctionTool
from google.adk.tools.load_web_page import load_web_page  # Example ADK tool

# ----------------------------------------------------
# ADK <-> MCP Conversion Utility
# ----------------------------------------------------
from google.adk.tools.mcp_tool.conversion_utils import adk_to_mcp_tool_type

# ----------------------------------------------------
# Prepare the ADK tool to expose via MCP
# ----------------------------------------------------
print("Initializing ADK load_web_page tool...")
adk_tool_to_expose = FunctionTool(load_web_page)
print(f"ADK tool '{adk_tool_to_expose.name}' initialized and ready to be exposed via MCP.")

# ----------------------------------------------------
# MCP Server Setup
# ----------------------------------------------------
print("Creating MCP Server instance...")
app = Server("adk-tool-exposing-mcp-server")

# ----------------------------------------------------
# Handler for MCP list_tools request
# ----------------------------------------------------
@app.list_tools()
async def list_mcp_tools() -> list[mcp_types.Tool]:
    """Handle MCP list_tools request and return available tools."""
    print("MCP Server: Received list_tools request.")

    mcp_tool_schema = adk_to_mcp_tool_type(adk_tool_to_expose)
    print(f"MCP Server: Advertising tool: {mcp_tool_schema.name}")

    return [mcp_tool_schema]

# ----------------------------------------------------
# Handler for MCP call_tool request
# ----------------------------------------------------
@app.call_tool()
async def call_mcp_tool(name: str, arguments: dict) -> list[mcp_types.Content]:
    """Handle MCP call_tool request and execute the requested tool."""
    print(f"MCP Server: Received call_tool request for '{name}' with args: {arguments}")

    if name == adk_tool_to_expose.name:
        try:
            # Execute the ADK tool asynchronously
            adk_tool_response = await adk_tool_to_expose.run_async(
                args=arguments,
                tool_context=None,  # Not using full ADK Runner context
            )

            print(f"MCP Server: ADK tool '{name}' executed successfully.")

            # Convert response to JSON string for MCP
            response_text = json.dumps(adk_tool_response, indent=2)

            return [
                mcp_types.TextContent(
                    type="text",
                    text=response_text
                )
            ]

        except Exception as e:
            print(f"MCP Server: Error executing ADK tool '{name}': {e}")

            error_text = json.dumps(
                {"error": f"Failed to execute tool '{name}': {str(e)}"}
            )

            return [
                mcp_types.TextContent(
                    type="text",
                    text=error_text
                )
            ]

    else:
        print(f"MCP Server: Tool '{name}' not found.")

        error_text = json.dumps(
            {"error": f"Tool '{name}' not implemented by this server."}
        )

        return [
            mcp_types.TextContent(
                type="text",
                text=error_text
            )
        ]

# ----------------------------------------------------
# MCP Server Runner
# ----------------------------------------------------
async def run_mcp_stdio_server():
    """Run the MCP server using STDIO transport."""
    async with mcp.server.stdio.stdio_server() as (read_stream, write_stream):
        print("MCP Stdio Server: Starting handshake with client...")

        await app.run(
            read_stream,
            write_stream,
            InitializationOptions(
                server_name=app.name,
                server_version="0.1.0",
                capabilities=app.get_capabilities(
                    notification_options=NotificationOptions(),
                    experimental_capabilities={},
                ),
            ),
        )

        print("MCP Stdio Server: Client disconnected.")

# ----------------------------------------------------
# Script Entry Point
# ----------------------------------------------------
if __name__ == "__main__":
    print("Launching MCP Server to expose ADK tools via stdio...")

    try:
        asyncio.run(run_mcp_stdio_server())

    except KeyboardInterrupt:
        print("\nMCP Server stopped by user.")

    except Exception as e:
        print(f"MCP Server encountered an error: {e}")

    finally:
        print("MCP Server process exiting.")


Initializing ADK load_web_page tool...
ADK tool 'load_web_page' initialized and ready to be exposed via MCP.
Creating MCP Server instance...
Launching MCP Server to expose ADK tools via stdio...
MCP Server encountered an error: asyncio.run() cannot be called from a running event loop
MCP Server process exiting.


C:\Users\DELL\AppData\Local\Temp\ipykernel_22612\447076239.py:148: RuntimeWarning: coroutine 'run_mcp_stdio_server' was never awaited
  print(f"MCP Server encountered an error: {e}")


- **Step 3: Test Your Custom MCP Server with an ADK Agent :** Now, create an ADK agent that will act as a client to the MCP Server you just built. This ADK agent will use `McpToolset` to connect to your `my_adk_mcp_server.py` script.
- Create an `agent.py` (e.g., in `./adk_agent_samples/mcp_client_agent/agent.py`):


In [ ]:
# agent.py
import os
from google.adk.agents import LlmAgent
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from mcp import StdioServerParameters
from config import config

# -----------------------------
# Setup API Key and model
# -----------------------------
APP_NAME = "chatbot_mcp"
USER_ID = "user_001"
SESSION_ID = "session_001"
MODEL = config.MODEL

google_api_key = config.GOOGLE_API_KEY.strip()

# set environment variables
os.environ["GOOGLE_API_KEY"] = google_api_key

PATH_TO_YOUR_MCP_SERVER_SCRIPT = "D:\Agent-Development-Kit\adk22\MCPServer\adk_mcp_server.py"

root_agent = LlmAgent(
    model=MODEL,
    name="web_reader_mcp_client_agent",
    instruction="Use the 'load_web_page' tool to fetch content from a URL provided by the user.",
    tools=[
        McpToolset(
            connection_params=StdioConnectionParams(
                server_params=StdioServerParameters(
                    command="python3",
                    args=[PATH_TO_YOUR_MCP_SERVER_SCRIPT]
                )
            ),
            # tool_filter=['load_web_page'] # Optional
        )
    ]
)


- Add an `__init__.py` in the same directory:
  ```python
  # ./adk_agent_samples/mcp_client_agent/__init__.py
  from . import agent
  ```
- **To Run the Test :**
  - **Start your custom MCP Server (optional, for separate observation):**
     - You can run your `my_adk_mcp_server.py` directly in one terminal to see its logs: `python my_adk_mcp_server.py`
     - It will print "Launching MCP Server..." and wait. The ADK agent (run via `adk web`) will then connect to this process if the `command` in `StdioConnectionParams` is set up to execute it.
  - **Run `adk web` for the client agent :**
     - Navigate to the parent directory of `mcp_client_agent` (e.g., `adk_agent_samples`) and run:
       ```powershell
       cd ./adk_agent_samples # Or your equivalent parent directory
       adk web
       ```
  - **Interact in the ADK Web UI :**
    - Select the `web_reader_mcp_client_agent`.
    - Try a prompt like: "Load the content from _https://example.com_"
- The ADK Agent (`web_reader_mcp_client_agent`) will use `McpToolset` to start and connect to your `my_adk_mcp_server.py`. Your MCP Server will receive the `call_tool` request, execute the ADK `load_web_page` tool, and return the result.
- The ADK agent will then relay this information. You should see logs from both the ADK Web UI (and its terminal) and potentially from your `my_adk_mcp_server.py` terminal you ran it separately.